In [0]:
from pyspark.sql import functions as F
df_dami = spark.table("cip_catalog.cleansed_dami_owner.employees")
df_parties = spark.table("cip_catalog.cleansed_dwh_owner.parties")
df_professions = spark.table("cip_catalog.cleansed_dwh_owner.person_professions")
df_employees = spark.table("cip_catalog.cleansed_dwh_owner.employees")

df = df_dami.alias("dami").join(df_parties, "PT_UNIFIED_KEY", "left_outer") \
                .join(df_employees.alias("employees"), ["PT_KEY", "PTORI_KEY"], "inner") \
                .join(df_professions.alias("professions"), "PROF_KEY", "left_outer") \
                .join(df_employees.alias("managers"), F.col("employees.EMP_MANAGER_KEY") == F.col("managers.EMP_KEY"), "left_outer") \
                    .select(
                        F.col("dami.PT_SOURCE_ID").alias("employee_CEN"),
                        F.col("professions.PROF_SHORT_DESC").alias("employee_profession"),
                        F.col("managers.PT_KEY").alias("PT_KEY"),
                        F.col("managers.PTORI_KEY").alias("PTORI_KEY")  
                    ) \
                .join(df_parties, ["PT_KEY", "PTORI_KEY"], "left_outer") \
                .join(df_dami.alias("dami_manager"), "PT_UNIFIED_KEY", "left_outer") \
                    .select(
                        F.col("employee_CEN"),
                        F.col("employee_profession"),
                        F.col("dami_manager.PT_SOURCE_ID").alias("manager_CEN") 
                    )

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("mifid_catalog.mifid_employees.mifid_employees")